In [1]:
# This version changes the plant tasks that explicitly disclose the potential computer vision model to be used (e.g., classification) 
from agents import manager, user_proxy

/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/fchen2/RDS/anaconda3/envs/phenoassistant_org/lib/python3.11/site-packages/

PhenoAssistant's available tools:
Tool 1: Name: perform_anova, Description: Perform Mixed-design Repeated Measures ANOVA on given data (Greenhouse-Geisser correction will be automatically applied if needed)
Tool 2: Name: perform_tukey_test, Description: Perform Post-hoc Tukey-Kramer test on given data
Tool 3: Name: extract_pipeline, Description: Extract and save a reproducible pipeline from chat history. Ask user to provide a name for the pipeline.
Tool 4: Name: get_pipeline_zoo, Description: Get the information of all registered pipelines. It is useful when a user wants to know what pipelines are available before executing any.
Tool 5: Name: get_pipeline_info, Description: Get the information of a specific pipeline. This is useful for you to know how to use a pipeline selected by the user, including the description, arguments, and output type.
Tool 6: Name: execute_pipeline, Description: Execute a saved pipeline from the pipeline zoo. Before executing a pipeline, you must call 'get_pi

/remote/rds/users/fchen2/codes/PhenoAssistant_Github_Revision/agents.py:577: RuntimeWarning: [PhenoAssistant] User tools auto-registered from ./functions
  warnings.warn("[PhenoAssistant] User tools auto-registered from ./functions", RuntimeWarning)


In [13]:
# creating plant tasks and model zoo and shuffle
plant_tasks = [
"Assess whether these potato plants appear healthy or illed.",
"Determine the ripening progress of these tomato fruits.",
"Determine the stress level (mild or severe) of the given rice plants.",
"What type of pest damage can be found in these cabbage leaves?",
"How mature these soybean pods appear?",
"Do these tomato plants appear under abiotic or biotic stress?",
"Recognise and record the species of these wildflowers.",
"Can you estimate the leaf greenness of these lettuces?",
"Compute the above-ground biomass for these barley images.",
"Infer chlorophyll concentration of these spinach leaves.",
"Quantify structural traits related to vine extension in these cucumber images.",
"Rank the maize individuals by vigour on a continuous scale.",
"How badly are these sunflowers affected by disease? Provide a score for each image.",
"Score the maturity of blueberries presented in these images.",
"Identify all the wheat spikes in these images.",
"Can you compute the leaf area of these Arabidopsis plants?",
"Map non-crop vegetation appearing in these wheat field photographs.",
"Pod counting for peas.",
"Highlight all the clusters in these grape images.",
"Delineate the crop rows in these soybean field images.",
]

models = {"instance-segmentation": 
            [
                "wheat_spike-instance-segmentation_m2fb",
                "arabidopsis_leaf-instance-segmentation_m2fb",
                "wheat_weed-instance-segmentation_m2fb",
                "grape_cluster-instance-segmentation_m2fb",
                "tomato_fruit-instance-segmentation_m2fb", # decoy
                "cabbage_leaf-pest-instance-segmentation_m2fb", # decoy
                "soybean_pod-instance-segmentation_m2fb", # decoy
                "wildflower_cluster-instance-segmentation_m2fb", # decoy
                "wheat_stem-instance-segmentation_m2fb", # decoy
                "wheat_leaf-instance-segmentation_m2fb", # decoy
                ], 
            "image-classification": 
            [
                "potato_health-disease-classify_dino2b",
                "tomato_fruit-ripening-stage-classify_dino2b",
                "rice_stress-level-classify_dino2b",
                "cabbage_leaf-pest-damage-classify_dino2b",
                "soybean_pod-maturity-status-classify_dino2b",
                "wildflower_species-classify_dino2b",
                "rice_disease-classify_dino2b", # decoy
                "cucumber_disease-classify_dino2b", # decoy
                "maize_vigour-classify_dino2b", # decoy
                "sunflower_disease-classify_dino2b", # decoy
                ], 
            "image-regression": 
            [
                "lettuce_leaf-greenness-regress_dino2b",
                "barley_biomass-regress_dino2b",
                "cucumber_vine-length-regress_dino2b",
                "maize_vigour-regress_dino2b",
                "sunflower_disease-severity-regress_dino2b",
                "potato_disease-severity-regress_dino2b", # decoy
                "arabidopsis_leaf-greenness-regress_dino2b", # decoy
                "rice_biomass-regress_dino2b", # decoy
                "arabidopsis_disease-severity-regress_dino2b", # decoy
                "grape_cluster-number-regress_dino2b", # decoy
                ]}

import random
random.seed(42)

random.shuffle(plant_tasks)
for category, model_list in models.items():
    random.shuffle(model_list)

In [20]:
# show shuffled plant tasks
plant_tasks

['Delineate the crop rows in these soybean field images.',
 'Do these tomato plants appear under abiotic or biotic stress?',
 'Identify all the wheat spikes in these images.',
 'How mature these soybean pods appear?',
 'Infer chlorophyll concentration of these spinach leaves.',
 'Score the maturity of blueberries presented in these images.',
 'Can you compute the leaf area of these Arabidopsis plants?',
 'Highlight all the clusters in these grape images.',
 'Recognise and record the species of these wildflowers.',
 'How badly are these sunflowers affected by disease? Provide a score for each image.',
 'Pod counting for peas.',
 'Quantify structural traits related to vine extension in these cucumber images.',
 'Determine the ripening progress of these tomato fruits.',
 'Rank the maize individuals by vigour on a continuous scale.',
 'Determine the stress level (mild or severe) of the given rice plants.',
 'Map non-crop vegetation appearing in these wheat field photographs.',
 'Can you es

In [21]:
# show shuffled model zoo
models

{'instance-segmentation': ['wildflower_cluster-instance-segmentation_m2fb',
  'cabbage_leaf-pest-instance-segmentation_m2fb',
  'wheat_spike-instance-segmentation_m2fb',
  'wheat_weed-instance-segmentation_m2fb',
  'tomato_fruit-instance-segmentation_m2fb',
  'wheat_leaf-instance-segmentation_m2fb',
  'arabidopsis_leaf-instance-segmentation_m2fb',
  'soybean_pod-instance-segmentation_m2fb',
  'grape_cluster-instance-segmentation_m2fb',
  'wheat_stem-instance-segmentation_m2fb'],
 'image-classification': ['cucumber_disease-classify_dino2b',
  'maize_vigour-classify_dino2b',
  'cabbage_leaf-pest-damage-classify_dino2b',
  'potato_health-disease-classify_dino2b',
  'rice_stress-level-classify_dino2b',
  'sunflower_disease-classify_dino2b',
  'tomato_fruit-ripening-stage-classify_dino2b',
  'soybean_pod-maturity-status-classify_dino2b',
  'wildflower_species-classify_dino2b',
  'rice_disease-classify_dino2b'],
 'image-regression': ['grape_cluster-number-regress_dino2b',
  'arabidopsis_leaf

In [19]:
# create the prompt
task = f'''For the following plant phenotyping tasks, choose the most suitable model from the model zoo that can effectively address each task.

For each plant phenotyping task, output your answer as "task: selected vision model".

Plant phenotyping tasks:
{plant_tasks}

Model zoo:
{models}
'''

res = user_proxy.initiate_chat(recipient=manager, message=task)

Admin (to manager):

For the following plant phenotyping tasks, choose the most suitable model from the model zoo that can effectively address each task.

For each plant phenotyping task, output your answer as "task: selected vision model"

Plant phenotyping tasks:
['Delineate the crop rows in these soybean field images.', 'Do these tomato plants appear under abiotic or biotic stress?', 'Identify all the wheat spikes in these images.', 'How mature these soybean pods appear?', 'Infer chlorophyll concentration of these spinach leaves.', 'Score the maturity of blueberries presented in these images.', 'Can you compute the leaf area of these Arabidopsis plants?', 'Highlight all the clusters in these grape images.', 'Recognise and record the species of these wildflowers.', 'How badly are these sunflowers affected by disease? Provide a score for each image.', 'Pod counting for peas.', 'Quantify structural traits related to vine extension in these cucumber images.', 'Determine the ripening pro

In [ ]:
# Ground truh mapping
# # classify
# 1. Assess whether these potato plants appear healthy or illed.: `potato_health-disease-classify_dino2b`
# 2. Determine the ripening progress of these tomato fruits.: `tomato_fruit-ripening-stage-classify_dino2b`
# 3. Determine the stress level (mild or severe) of the given rice plants.: `rice_stress-level-classify_dino2b`
# 4. What type of pest damage can be found in these cabbage leaves?: `cabbage_leaf-pest-damage-classify_dino2b`
# 5. How mature these soybean pods appear?: `soybean_pod-maturity-status-classify_dino2b`
# 6. Do these tomato plants appear under abiotic or biotic stress?: NOT in model zoo, suggest classification
# 7. Recognise and record the species of these wildflowers.: `wildflower_species-classify_dino2b`

# # regress
# 8. Can you estimate the leaf greenness of these lettuces?: `lettuce_leaf-greenness-regress_dino2b`
# 9. Compute the above-ground biomass for these barley images.: `barley_biomass-regress_dino2b`
# 10. Infer chlorophyll concentration of these spinach leaves.: NOT in model zoo, suggest regression
# 11. Quantify structural traits related to vine extension in these cucumber images.: `cucumber_vine-length-regress_dino2b`
# 12. Rank the maize individuals by vigour on a continuous scale.: `maize_vigour-regress_dino2b`
# 13. How badly are these sunflowers affected by disease? Provide a score for each image.: `sunflower_disease-severity-regress_dino2b`
# 14. Score the maturity of blueberries presented in these images.: NOT in model zoo, suggest regression or classification

# # ins seg
# 15. Identify all the wheat spikes in these images.: `wheat_spike-instance-segmentation_m2fb`
# 16. Can you compute the leaf area of these Arabidopsis plants?: `arabidopsis_leaf-instance-segmentation_m2fb`
# 17. Map non-crop vegetation appearing in these wheat field photographs.: `wheat_weed-instance-segmentation_m2fb`
# 18. Pod counting for peas.: NOT in model zoo, suggest regression or instance segmentation
# 19. Highlight all the clusters in these grape images.: `grape_cluster-instance-segmentation_m2fb`
# 20. Delineate the crop rows in these soybean field images.: NOT in model zoo, suggest instance segmentation